## 编码与解码文本
前面说过，文本编码 (Encoding) 过程包含两个步骤：
1. 分词：使用分词器按某种策略将文本切分为 tokens；
2. 映射：将 tokens 转化为对应的 token IDs。

In [1]:
from transformers import AutoTokenizer

root = "/pfs/pfs-UPi9IY/sdata/chongqing/code/python-libraries/model"
model_name="bert-base-cased"#模型名称
cache_dir=f"{root}/{model_name}"#模型保存路径
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
# tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

/pfs/pfs-UPi9IY/sdata/chongqing/anaconda3/envs/lerobot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


可以看到，BERT 分词器采用的是子词切分策略，它会不断切分词语直到获得词表中的 token，例如 “transformer” 会被切分为 “transform” 和 “##er”。
然后，我们通过 convert_tokens_to_ids() 将切分出的 tokens 转换为对应的 token IDs：

In [ ]:
ids = tokenizer.convert_tokens_to_ids(tokens)

print(ids)

## 处理多段文本
实际场景中，我们应该直接使用分词器对文本进行处理，

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
# tokenizer = AutoTokenizer.from_pretrained(checkpoint)

root = "/pfstem/chongqing/code/python-libraries/model"
model_name="distilbert-base-uncased-finetuned-sst-2-english"#模型名称
cache_dir=f"{root}/{model_name}"#模型保存路径
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_name, cache_dir=cache_dir)

sequence = "I've been waiting for a HuggingFace course my whole life."

tokenized_inputs = tokenizer(sequence, return_tensors="pt")
print("Inputs Keys:\n", tokenized_inputs.keys())
print("\nInput IDs:\n", tokenized_inputs["input_ids"])

output = model(**tokenized_inputs)
print("\nLogits:\n", output.logits)

/root/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Inputs Keys:
 dict_keys(['input_ids', 'attention_mask'])

Input IDs:
 tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]])

Logits:
 tensor([[-1.5607,  1.6123]], grad_fn=<AddmmBackward0>)


### AutoTokenizer常用配置参数


In [8]:
from transformers import AutoTokenizer

# checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
# tokenizer = AutoTokenizer.from_pretrained(checkpoint)
root = "/pfstem/chongqing/code/python-libraries/model"
model_name="distilbert-base-uncased-finetuned-sst-2-english"#模型名称
cache_dir=f"{root}/{model_name}"#模型保存路径
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)

# sequences = [
#     "I've been waiting for a HuggingFace course my whole life.", 
#     "So have I!"
# ]
sequences = ["I've been waiting for a HuggingFace course my whole life."]

model_inputs = tokenizer(sequences, 
                        padding="max_length",    # padding="max_length"：将所有序列填充到模型能够接受的最大长度，例如 BERT 模型就是 512。
                        padding_side="right",    # padding_side="right"：从右侧开始填充。
                        max_length=8,           # max_length=8：设置最大长度为 8。
                        return_tensors="pt"       # return_tensors="pt"：返回的类型为 PyTorch 张量。
                        # , truncation=True # 默认是False
                        )
print(model_inputs)

{'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


## 添加 Token

### 添加新 token
* `add_tokens()` 添加普通 token：参数是新 token 列表，如果 token 不在词表中，就会被添加到词表的最后。
* `add_special_tokens()` 添加特殊 token：参数是包含特殊 token 的字典，键值只能从 bos_token, eos_token, unk_token, sep_token, pad_token, cls_token, mask_token, additional_special_tokens 中选择。同样地，如果 token 不在词表中，就会被添加到词表的最后。添加后，还可以通过特殊属性来访问这些 token，例如 tokenizer.cls_token 就指向 cls token。

In [10]:
from transformers import AutoTokenizer

root = "/pfstem/chongqing/code/python-libraries/model"
model_name="bert-base-cased"#模型名称
cache_dir=f"{root}/{model_name}"#模型保存路径
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)

In [11]:
# add_tokens() 添加普通 token：参数是新 token 列表，如果 token 不在词表中，就会被添加到词表的最后。
num_added_toks = tokenizer.add_tokens(["new_token1", "my_new-token2"])
print("We have added", num_added_toks, "tokens")

We have added 2 tokens


In [13]:
# add_special_tokens() 添加特殊 token：参数是包含特殊 token 的字典，键值只能从 bos_token, eos_token, unk_token, sep_token, pad_token, cls_token, mask_token, additional_special_tokens 中选择。
# 同样地，如果 token 不在词表中，就会被添加到词表的最后。添加后，还可以通过特殊属性来访问这些 token，例如 tokenizer.cls_token 就指向 cls token。
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
special_tokens_dict = {"cls_token": "[MY_CLS]"}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print("We have added", num_added_toks, "special tokens")
print(tokenizer.cls_token)

# 删除 token
# tokenizer.remove_tokens(["[MY_CLS]"])


We have added 1 special tokens
[MY_CLS]


### 调整 embedding 矩阵
向词表中添加新 token 后，必须重置模型 embedding 矩阵的大小，也就是向矩阵中添加新 token 对应的 embedding，这样模型才可以正常工作，将 token 映射到对应的 embedding。

In [15]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(model_name, cache_dir=cache_dir)

print('vocabulary size:', len(tokenizer))
num_added_toks = tokenizer.add_tokens(['[ENT_START]', '[ENT_END]'], special_tokens=True)
print("After we add", num_added_toks, "tokens")
print('vocabulary size:', len(tokenizer))


model.resize_token_embeddings(len(tokenizer))
print(model.embeddings.word_embeddings.weight.size())

# Randomly generated matrix
print(model.embeddings.word_embeddings.weight[-2:, :])

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


vocabulary size: 28996
After we add 2 tokens
vocabulary size: 28998
torch.Size([28998, 768])
tensor([[-0.0081, -0.0199, -0.0089,  ..., -0.0140, -0.0205, -0.0110],
        [-0.0081, -0.0199, -0.0088,  ..., -0.0140, -0.0205, -0.0110]],
       grad_fn=<SliceBackward0>)
